In [1]:
# run <export OPENAI_API_KEY=sk-WXT******************************Q8> in CLI

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("training.csv")
df.fillna("", inplace=True)
# this training.csv consists of hand-curated prompts and completions

In [3]:
df

,source_columns,destination_column,transformation,additional_remarks,ground_truth
0,BENE_FIPS_STATE_CD,pst,Map\nAR to Arkansas\nCT to Connecticut\nDC to ...,,case when BENE_FIPS_STATE_CD= 'AR' then 'Arkan...
1,CLM_LINE_NUM,clid,,,CLM_LINE_NUM as clid
2,ALPHA_CODE,ftst,Coalesce with '' to remove any NULL values,,"coalesce(ALPHA_CODE,'') as ftst"
3,BENE_PTNT_STUS_CD,drdn,Mapper to be put here -\n02=Discharged/transfe...,,case when BENE_PTNT_STUS_CD = '02' then 'Disch...
4,CLM_FROM_DT,efdt,"Date should not be blank, or '1000-01-01' or '...",,"CASE WHEN CLM_FROM_DT NOT IN ('','1000-01-01',..."
...,...,...,...,...,...
67,BENE_DEATH_DT,df,"If not '' or '1000-01-01' or '9999-12-31, then...",,"CASE WHEN BENE_DEATH_DT NOT IN ('','1000-01-01..."
68,PIPELINE_ID,pipid,No transformation,,PIPELINE_ID as pipid
69,SFN,rcdt,Use regex to extract month,Regex - 'D\\d+',"date_trunc('month',to_date(REPLACE(regexp_subs..."
70,BENE_HIC_NUM,pthcno,Remove null if exists,,"COALESCE(BENE_HIC_NUM,'') as pthcno"


In [4]:
prompt = []
completion = []

for index, row in df.iterrows():

    source_columns = row.source_columns
    destination_column = row.destination_column
    transformation = row.transformation if len(row.transformation) > 0 else "No transformation"
    additional_remarks = row.additional_remarks
    ground_truth = f" select {row.ground_truth} from your_table; ###"

    prompt_item = f"""Write a SQL query for transforming {source_columns} into {destination_column}, and apply the following transformation:\n{transformation}. {additional_remarks}\n\n###\n\n"""

    prompt.append(prompt_item)
    completion.append(ground_truth)

In [ ]:
# for index, row in df.iterrows():

#     source_columns = row.source_columns
#     destination_column = row.destination_column
#     transformation = row.transformation if len(row.transformation) > 0 else "No transformation"
#     additional_remarks = row.additional_remarks
#     ground_truth = f" select {row.ground_truth} from your_table; ###"

#     prompt_item = f"""Write a SQL query for transforming {source_columns} into {destination_column}, and apply the following transformation:\n{transformation}\n{additional_remarks}"""
#     p1 = "The output query should be like - \"select <prediction here> from your_table;\""
    
#     if len(additional_remarks)>0:
#         prompt_item = f"""{prompt_item}\n{additional_remarks}\n\n{p1}\n\n###\n\n"""
#     else:
#         prompt_item = f"""{prompt_item}\n\n{p1}\n\n###\n\n"""

#     prompt.append(prompt_item)
#     completion.append(ground_truth)

In [5]:
ground_truth

" select coalesce(CDSCL,'') as pdn from your_table; ###"

In [6]:
print(prompt_item)

Write a SQL query for transforming CDSCL into pdn, and apply the following transformation:
Coalesce with ''. 

###




In [7]:
processed_data = pd.DataFrame(
    {
        "prompt": prompt,
        "completion": completion,
    }
)

In [8]:
processed_data

,prompt,completion
0,Write a SQL query for transforming BENE_FIPS_S...,select case when BENE_FIPS_STATE_CD= 'AR' the...
1,Write a SQL query for transforming CLM_LINE_NU...,select CLM_LINE_NUM as clid from your_table; ###
2,Write a SQL query for transforming ALPHA_CODE ...,"select coalesce(ALPHA_CODE,'') as ftst from y..."
3,Write a SQL query for transforming BENE_PTNT_S...,select case when BENE_PTNT_STUS_CD = '02' the...
4,Write a SQL query for transforming CLM_FROM_DT...,"select CASE WHEN CLM_FROM_DT NOT IN ('','1000..."
...,...,...
67,Write a SQL query for transforming BENE_DEATH_...,"select CASE WHEN BENE_DEATH_DT NOT IN ('','10..."
68,Write a SQL query for transforming PIPELINE_ID...,select PIPELINE_ID as pipid from your_table; ###
69,Write a SQL query for transforming SFN into rc...,"select date_trunc('month',to_date(REPLACE(reg..."
70,Write a SQL query for transforming BENE_HIC_NU...,"select COALESCE(BENE_HIC_NUM,'') as pthcno fr..."


In [9]:
processed_data.to_csv("processed_data_gpt35.csv", index=False)

## Fine Tuning

In [11]:
import subprocess
import openai
import os

In [14]:
# convert .csv to .jsonl
# subprocess.run('openai tools fine_tunes.prepare_data --file processed_data.csv --quiet'.split())

# RUN THIS IN CLI INSTEAD! - openai tools fine_tunes.prepare_data -f processed_data.csv

CompletedProcess(args=['openai', 'tools', 'fine_tunes.prepare_data', '--file', 'training.csv', '--quiet'], returncode=1)

In [60]:
openai.api_key = "YOUR_OPENAI_API_KEY"

In [85]:
file_response = openai.File.create(
  file=open("processed_data_old_prepared.jsonl", "rb"),
  purpose='fine-tune'
)

In [87]:
new_file_id = "file-fdNxWJzJ3RNDSeKbkhQxNNLI"
old_file_id = "file-NSXSytLd9YnAROtPFjhmUnYK"

In [86]:
openai.File.list()

<OpenAIObject list at 0x28640fa3450> JSON: {
  "data": [
    {
      "bytes": 32371,
      "created_at": 1680338559,
      "filename": "file",
      "id": "file-fdNxWJzJ3RNDSeKbkhQxNNLI",
      "object": "file",
      "purpose": "fine-tune",
      "status": "processed",
      "status_details": null
    },
    {
      "bytes": 20348,
      "created_at": 1680339751,
      "filename": "file",
      "id": "file-NSXSytLd9YnAROtPFjhmUnYK",
      "object": "file",
      "purpose": "fine-tune",
      "status": "processed",
      "status_details": null
    }
  ],
  "object": "list"
}

In [88]:
file_response['id']

'file-NSXSytLd9YnAROtPFjhmUnYK'

In [127]:
res_fine_tune = openai.FineTune.create(training_file=old_file_id)

In [128]:
fine_tune_queued_model_id = res_fine_tune['id']
fine_tune_queued_model_id

'ft-CyorF9NO9XPA1ShgRWdAg34j'

In [ ]:
import openai


In [13]:
fine_tune_list = openai.FineTune.list()
print(fine_tune_list['data'][0]['status'])
print(fine_tune_list['data'][1]['status'])
print(fine_tune_list['data'][2]['status'])

cancelled
cancelled
succeeded


In [131]:
# openai.FineTune.cancel(id="ft-2pBlX9SLtmkMB9AJn4TCoceA")

In [110]:
fine_tune_list

<OpenAIObject list at 0x28641b51bd0> JSON: {
  "data": [
    {
      "created_at": 1680338565,
      "fine_tuned_model": null,
      "hyperparams": {
        "batch_size": null,
        "learning_rate_multiplier": null,
        "n_epochs": 4,
        "prompt_loss_weight": 0.01
      },
      "id": "ft-2pBlX9SLtmkMB9AJn4TCoceA",
      "model": "curie",
      "object": "fine-tune",
      "organization_id": "org-vwgp7xyYLaLY7qgK36xyyU0L",
      "result_files": [],
      "status": "pending",
      "training_files": [
        {
          "bytes": 32371,
          "created_at": 1680338559,
          "filename": "file",
          "id": "file-fdNxWJzJ3RNDSeKbkhQxNNLI",
          "object": "file",
          "purpose": "fine-tune",
          "status": "processed",
          "status_details": null
        }
      ],
      "updated_at": 1680338565,
      "validation_files": []
    },
    {
      "created_at": 1680339806,
      "fine_tuned_model": null,
      "hyperparams": {
        "batch_size": 

In [261]:
fine_tuned_model = openai.FineTune.retrieve(id=fine_tune_queued_model_id)

In [262]:
fine_tuned_model['fine_tuned_model']

In [115]:
fine_tuned_model['fine_tuned_model']

In [52]:
source_columns = 'col1'
destination_column = 'col2'
transformation = """Convert to DD-MM-YYYY date format if value is not in (01-01-1000, 31-12-9999)"""
additional_remarks = ""


prompt_item = f"""Write a SQL query for transforming {source_columns} into {destination_column}, and apply the following transformation:\n{transformation}\n{additional_remarks}"""
p1 = "The output query should be like - \"select <prediction here> from your_table;\""

if len(additional_remarks)>0:
    prompt_item = f"""{prompt_item}\n{additional_remarks}\n\n{p1}\n\n###\n\n"""
else:
    prompt_item = f"""{prompt_item}\n\n{p1}\n\n###\n\n"""

In [53]:
print(prompt_item)

Write a SQL query for transforming col1 into col2, and apply the following transformation:
Convert to DD-MM-YYYY date format if value is not in (01-01-1000, 31-12-9999)


The output query should be like - "select <prediction here> from your_table;"

###




In [12]:
import openai

openai.api_key = "YOUR_OPENAI_API_KEY"

In [58]:
response = openai.Completion.create(
  model= "davinci",
  prompt=prompt_item,
  temperature=0.7,
  max_tokens=256,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  stop="###"
)

In [59]:
response

<OpenAIObject text_completion id=cmpl-70R5SK5dFbnUp4hNsKsiaiyCcb1Je at 0x28642ca04a0> JSON: {
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "text": "\nThe following table is used in this blog post.\n\ncreate table your_table ( col1 date , col2 date ); insert into your_table values ( '01-01-1990' , '01-01-1990' ); insert into your_table values ( '01-01-1990' , '08-08-2001' ); insert into your_table values ( '01-01-1990' , '08-08-2001' ); insert into your_table values ( '01-01-1990' , '01-01-2000' ); insert into your_table values ( '01-01-1990' , '01-01-2000' ); insert into your_table values ( '01-01-1990' , '08-08-2001' ); insert into your_table values ( '01-01-1999' , '08-08-2001' ); insert into your_table values ( '01-01-1999' , '01-01-2000' ); insert into your_table values ( '01-01-1999' , '01-01-2000' ); insert into your_table values ( '01-01-1999' , '08-08-2001'"
    }
  ],
  "created": 1680338478,
  "id": "cmpl-70R5SK5dFbnUp4

In [31]:
print(response['choices'][0]['text'])

 case when col1 in ('01-01-1000','31-12-9999') then ('dd-mm-yyyy') else ('dd-mm-yyyy'||'') end as col2


In [143]:
response

<OpenAIObject text_completion id=cmpl-70BPL8XATRzC9D35hOJH7hB2HBUSV at 0x1e6c3d650e0> JSON: {
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "text": " case when clm_efctv_dt not in ('','1000-01-01','9999-12-31')  then (to_date(clm_efctv_dt,'yyyy-mm-dd')) end as cpdt###\n\n case when clm_efctv_dt not in ('','1000-01-01','9999-12-31')  then (to_date(clm_efctv_dt,'yyyy-mm-dd')) end as cpdt######################################################################################################################################################################################################################################################################################################################################################################################################################################################"
    }
  ],
  "created": 1680278207,
  "id": "cmpl-70BPL8XATRzC9D35hOJH7hB2HBUSV",
  "model": "curie:ft-personal-2023

In [60]:
content = openai.File.retrieve("file-kAEOiVz5JIsNgKWCEBcXl7qZ")

In [61]:
content

<File file id=file-kAEOiVz5JIsNgKWCEBcXl7qZ at 0x1e6c3cb07c0> JSON: {
  "bytes": 14154,
  "created_at": 1680273440,
  "filename": "compiled_results.csv",
  "id": "file-kAEOiVz5JIsNgKWCEBcXl7qZ",
  "object": "file",
  "purpose": "fine-tune-results",
  "status": "processed",
  "status_details": null
}